## 01 — What We Built

Before moving on to the library version, we take stock.

Good engineers do not just ship and move on — they understand what they built, where it breaks, and what it would cost to fix those things. This notebook measures the handbuilt system honestly.

## What We Built — Component Inventory

| Component | Where | What it does |
|-----------|-------|--------------|
| **Douglas-Peucker** | Module 01 | Reduces point count per line segment |
| **LOD pipeline** | Module 02 | Produces 4 simplified GeoJSON files |
| **Bbox computation** | Module 03 | Gets the extent of any feature |
| **Bbox intersection** | Module 03 | Tests if a feature overlaps the viewport |
| **Uniform grid index** | Module 04 | Buckets features for fast viewport queries |
| **LOD decision function** | Module 05 | Selects the right file by zoom level |
| **Live map viewer** | Module 06 | Wires everything into an interactive display |

Each component was built from scratch. We understand every line.

## Measuring the System

In [5]:
import json
import time
from pathlib import Path

lod_dir  = Path("../../data/lod")
raw_path = Path("../../data/ne_10m_railroads.geojson")

lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

print(f"{'File':<28} {'Size (MB)':>10} {'Features':>10} {'Total pts':>12} {'Load (s)':>10}")
print("-" * 75)

for label, filename in [("original", None)] + list(lod_files.items()):
    path = raw_path if filename is None else lod_dir / filename
    t0 = time.perf_counter()
    with open(path) as f:
        data = json.load(f)
    load_time = time.perf_counter() - t0
    feats = data["features"]
    n_pts = sum(len(f["geometry"]["coordinates"]) for f in feats)
    size  = path.stat().st_size / 1_000_000
    print(f"{label:<28} {size:>10.2f} {len(feats):>10,} {n_pts:>12,} {load_time:>10.3f}")

File                          Size (MB)   Features    Total pts   Load (s)
---------------------------------------------------------------------------
original                          39.60     25,413    1,396,480      1.731
coarse                             1.08      2,845        5,690      0.026
medium                             9.66     25,413       55,516      0.259
fine                              11.34     25,413      124,844      0.346
extra_fine                        18.98     25,413      441,890      0.769


## Where the System Still Hurts

The viewer works. But it has real limitations. Let's name them honestly.

### Pain Point 1 — Startup Cost

Every session, we load 4 files and build 4 grid indexes. This takes several seconds before the map is usable.

A tile server has no startup cost — tiles are pre-built and stored. The server just reads a file from a database and sends it.

In [6]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

class GridIndex:
    CELL_SIZE = 10.0
    def __init__(self): self.cells = {}
    def _cells(self, bbox):
        lo, la, hi, ha = bbox; cs = self.CELL_SIZE
        return [(c, r) for c in range(int((lo+180)/cs), int((hi+180)/cs)+1)
                       for r in range(int((la+ 90)/cs), int((ha+ 90)/cs)+1)]
    def build(self, features):
        self.cells = {}
        for i, f in enumerate(features):
            for cell in self._cells(feature_bbox(f)): self.cells.setdefault(cell,[]).append((i,f))

total_startup = 0
for filename in lod_files.values():
    t0 = time.perf_counter()
    with open(lod_dir / filename) as f:
        feats = json.load(f)["features"]
    idx = GridIndex()
    idx.build(feats)
    elapsed = time.perf_counter() - t0
    total_startup += elapsed

print(f"Total startup time (load + index build): {total_startup:.2f}s")

Total startup time (load + index build): 2.49s


### Pain Point 2 — GeoJSON Is Verbose

GeoJSON is human-readable text. Every coordinate is stored as a decimal number string. A production mapping pipeline uses **binary encoding** (Mapbox Vector Tiles, MVT) which stores coordinates as integers relative to the tile origin — 5–10× smaller than equivalent GeoJSON and much faster to parse.

In [7]:
# Rough estimate: how large would our files be in a binary format?
# MVT stores coordinates as 2-byte integers per axis
# GeoJSON stores them as ~8-character float strings

GEOJSON_BYTES_PER_COORD = 16   # avg chars for [lon, lat] pair including punctuation
MVT_BYTES_PER_COORD     = 4    # 2 bytes each for x, y as zigzag-encoded varint

for filename in lod_files.values():
    with open(lod_dir / filename) as f:
        feats = json.load(f)["features"]
    total_pts = sum(len(f["geometry"]["coordinates"]) for f in feats)
    actual_mb  = (lod_dir / filename).stat().st_size / 1_000_000
    est_mvt_mb = total_pts * MVT_BYTES_PER_COORD / 1_000_000
    print(f"{filename:<38} actual: {actual_mb:.2f}MB   est MVT: {est_mvt_mb:.2f}MB")

railroads_coarse.geojson               actual: 1.08MB   est MVT: 0.02MB
railroads_medium.geojson               actual: 9.66MB   est MVT: 0.22MB
railroads_fine.geojson                 actual: 11.34MB   est MVT: 0.50MB
railroads_extra_fine.geojson           actual: 18.98MB   est MVT: 1.77MB


### Pain Point 3 — The Whole File Is Always Resident

To query the fine LOD for Paris, we load the entire `railroads_fine.geojson` into memory — including Australia, South America, and Russia. A tile system would read only the Paris tile from a database, never touching the rest.

Our grid index helps at query time, but the full file still had to load first.

### Pain Point 4 — No Partial Load or Streaming

When the user pans to a new region, we re-query the index immediately — but the data was all loaded at startup. A tile server streams only the tiles the user actually views. If the user never visits Australia, those tiles are never fetched.

## The Decision Inventory

Every system embeds design decisions. Here are ours, stated explicitly:

| Decision | What we chose | What we gave up |
|----------|--------------|------------------|
| File format | GeoJSON (text) | Binary efficiency |
| Simplification algorithm | Douglas-Peucker via Shapely | Topology-preserving alternatives |
| LOD levels | 4 fixed levels | Continuous zoom-adaptive detail |
| Coarse filter | scalerank ≤ 4 | Coverage in scalerank 5+ regions |
| Spatial index | Uniform 10° grid | Adaptive indexes (R-tree, quadtree) |
| Culling granularity | Feature bbox | True geometry intersection |
| Transition policy | Fixed zoom thresholds | Hysteresis (implemented but not used in final viewer) |
| Memory model | All data loaded at startup | Lazy / tile-based loading |

None of these decisions are wrong. They are appropriate for a teaching system built from scratch. A production system makes different choices for different reasons.

## Exercise A

Measure the peak memory usage of the viewer at startup (after all 4 LOD files are loaded and all 4 indexes are built).

Use Python's `tracemalloc` module:

```python
import tracemalloc
tracemalloc.start()
# ... load and build ...
current, peak = tracemalloc.get_traced_memory()
print(f"Peak memory: {peak / 1_000_000:.1f} MB")
```

How does this compare to just reading the four files without building indexes?

In [ ]:
# Measure peak memory: (a) loading files only, (b) loading + building indexes

import tracemalloc
import gc

def measure_load_only():
    loaded = {}

    tracemalloc.start()

    for filename in lod_files.values():
        with open(lod_dir / filename) as f:
            loaded[filename] = json.load(f)["features"]

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return peak, loaded


def measure_load_and_build():
    loaded = {}
    indexes = {}

    tracemalloc.start()

    for filename in lod_files.values():
        with open(lod_dir / filename) as f:
            feats = json.load(f)["features"]
        loaded[filename] = feats

        idx = GridIndex()
        idx.build(feats)
        indexes[filename] = idx

    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return peak, loaded, indexes


peak_load_only, loaded_only = measure_load_only()

del loaded_only
gc.collect()

peak_load_build, loaded_built, indexes_built = measure_load_and_build()

extra_mb = (peak_load_build - peak_load_only) / 1_000_000

print(f"Peak memory — files only:           {peak_load_only / 1_000_000:.1f} MB")
print(f"Peak memory — files + indexes:      {peak_load_build / 1_000_000:.1f} MB")
print(f"Extra peak memory for indexes:      {extra_mb:.1f} MB")

# Reading the four LOD files by themselves used about 208.3 MB of peak memory.
# Reading the files and then building all four grid indexes increased peak memory
# to about 213.5 MB.
#
# So the indexes added about 5.2 MB of extra peak memory.
# That comparison shows that most of the memory cost comes from loading and keeping
# the GeoJSON feature data, not from the grid indexes.
# The indexes do add overhead, but in this system that extra memory is relatively small.

Peak memory — files only:           208.3 MB
Peak memory — files + indexes:      213.5 MB
Extra peak memory for indexes:      5.2 MB


## Exercise B

Write a short summary (8–12 sentences) of the Railroad LOD system as if you were presenting it to a team that had never seen it.

Cover:
- What problem it solves
- What the four major components are
- What the main performance tradeoffs are
- What you would change if this needed to serve 10 million users instead of one notebook

Write it in the cell below as markdown.

This Railroad LOD system solves the problem of drawing a very large railroad dataset in an interactive map without using the full raw geometry at every zoom level. It does that by reducing detail when the user is zoomed out and showing more detail when the user is zoomed in. The first major component is the LOD pipeline, which produces four simplified GeoJSON files. The second is the bounding-box logic, which gives each feature an extent and tests whether it overlaps the current view. The third is the uniform grid index, which limits each query to nearby cells instead of scanning the whole dataset every time. The fourth is the zoom decision function, which chooses the correct LOD file for the current zoom level and connects that choice to the live viewer. The main performance tradeoffs are startup cost, verbose GeoJSON files, full-file memory residency, and no streaming. Those choices are reasonable for a teaching system, but they would not scale well to very large traffic. If this needed to serve 10 million users, I would move preprocessing to a server pipeline, use vector tiles instead of GeoJSON, and stream only the tiles needed for the current view.

## Check Your Understanding

We identified four pain points: startup cost, verbose format, whole-file loading, and no streaming.

Rank them from **most impactful** to **least impactful** for a user on a slow connection (e.g., mobile data). Justify your ranking in 2–3 sentences.

---

For a user on a slow connection, the most important pain point is whole-file loading, followed by no streaming, then verbose format, and finally startup cost. Whole-file loading is the biggest problem because the user has to download far more data than the current view actually needs, instead of just getting what is on screen. No streaming makes that worse because nothing can appear until the full file has arrived, and verbose GeoJSON increases the transfer size even more. Startup cost matters least here because on a slow connection the main bottleneck is already the network transfer.

## Next

In [Module 07 — The Library Version](../07-The_Library_Version/README.md), we hand the problem to `tippecanoe` and see how a professional tool addresses every one of these pain points.